# 第1章 — 情報セキュリティエンジニアのための機械学習入門

本書第1章（pp.1–12）を読みながら、Colabの準備と、学習から評価までの一巡を行う。
第2章に進む前に、特徴量、ラベル、訓練データ、テストデータの役割を区別しておきたい。
ページ番号はPDFビューアの通し番号ではなく、本文に印刷されたページ番号である。

| 本書 | このノートブックで扱うこと |
|---|---|
| §1.1、pp.1–2 | 検知結果を読み解くために機械学習を学ぶ理由 |
| §1.2、pp.2–10 | Colab、パッケージ、ランタイム、ファイルの保存 |
| §1.3、pp.10–12 | データを準備し、学習と評価を分ける手順 |

## §1.1 検知器が間違えたとき、何を調べるか

本書は、マルウェア検知、IDS、迷惑メール分類、UEBAを例に、セキュリティの仕事と機械学習の接点を説明している。
共通するのは、観測したデータから対象の性質を推定することだ。
たとえばメール本文から迷惑メールらしさを求めたり、通常時の通信と異なる振る舞いを探したりする。
前者のように正解ラベルを付けて学習する方法と、通常の分布からの逸脱を調べる方法では、用意するデータも評価方法も異なる。
第2章で実装するのは、正解付きのデータを使う**教師ありの二値分類**である。

本書が機械学習の知識を勧める理由は、検知エンジンを自作できるようになることだけではない。
正当なメールが隔離されたとき、その本文に含まれる何が判断に影響したのか。
既知の攻撃に似た入力を見逃したとき、そもそも違いを捉えられる特徴量があったのか。
こうした切り分けには、入力、学習に使った例、判定基準を順に調べる必要がある。

| 観測した事象 | 調べたいこと |
|---|---|
| 正当なメールが隔離される | どの表現や送信者に誤検知が偏っているか。閾値が厳しすぎないか |
| 攻撃メールが届いてしまう | 学習時に似た例があったか。本文以外の情報が必要ではないか |
| 実験では良いが運用では悪い | データの時期、送信者、攻撃の種類、攻撃の混入率が違わないか |

この段階では、検知結果が誤っていると分かっても、原因まで分かったことにはならない。
モデル、データ、運用上の判定を分けて調べる、という見方を持って実習を進める。

## §1.2 Colabで揃えるもの

Colabは、ブラウザ上のノートブックから、サーバー上のPythonを実行する環境である。
ノートブックには説明、コード、出力を保存できる。一方、変数や一時ファイルが存在するのは**ランタイム**の側だ。
ノートブックを保存することと、実行中の状態を保存することは別になる。

新しいCPUランタイムで **ランタイム → すべてのセルを実行** を選ぶ。
GPUやTPUへの切り替えは、本書§1.2.2で紹介されているが、今回の小規模な表形式・テキスト分類では使わない。
GPUを選ぶだけで、scikit-learnの処理が自動的にGPUへ移るわけでもない。

| パッケージ | この実習での担当 |
|---|---|
| NumPy / SciPy | 数値配列、数値計算、疎行列 |
| pandas | 特徴量、ラベル、集計結果を表として扱う |
| scikit-learn | 分割、前処理、分類器、評価指標 |
| Optuna | ハイパーパラメータの探索 |
| LightGBM | 第2章の迷惑メール分類器 |
| Matplotlib | 混同行列や特徴量重要度の表示 |

本書ではColabに導入済みのパッケージを利用し、不足分を`pip`で追加している（§1.2.3、pp.8–9）。
ここでは、実行時期によるAPIの違いを減らすため、使用する主要パッケージのバージョンを指定する。
TensorFlowとKerasは本書の後続章で使われるが、第1・2章の実行には不要である。

最初のコードセルは、ノートブックを動かしているPython自身の`pip`を呼び出す。
`check=True`にしてあるため、インストールが失敗した場合はその場で停止する。
別バージョンをすでに読み込んだセッションでは、インストール後にランタイムを再起動し、先頭から実行する。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[
            "numpy==2.2.6",
            "pandas==2.2.3",
            "scipy==1.15.3",
            "scikit-learn==1.7.2",
            "matplotlib==3.10.3",
            "optuna==4.5.0",
            "optuna-integration[lightgbm]==4.5.0",
            "lightgbm==4.6.0",
        ],
    ],
    check=True,
)

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt

print(f"Python {platform.python_version()} | CPU runtime")
for package in ("numpy", "pandas", "scipy", "scikit-learn", "optuna", "lightgbm"):
    print(f"{package}: {version(package)}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

### 実行順序と、保存されるもの

セルの見た目の順序と、実際に実行した順序は一致するとは限らない。
以前作った変数が残っていると、途中のセルだけ動いてしまい、他の人の環境では再現しないことがある。
「新しいランタイムで先頭から動くか」を確認するのは、この状態への依存を取り除くためである。

| 保存対象 | ランタイムが終了したあと |
|---|---|
| 保存した`.ipynb`の説明・コード・出力 | 保存先のノートブックに残る |
| Pythonの変数、インストールしたパッケージ | 新しいランタイムでは作り直す |
| ランタイム内に作成・ダウンロードしたファイル | 消えるため、必要な結果はダウンロードする |

本書§1.2.5（pp.9–10）にはセッションや維持時間の制限が紹介されている。
当時の時間・料金を現在の保証値とは扱わない。現在の条件は[Colab FAQ](https://research.google.com/colaboratory/faq.html)で確認する。
第2章では、実験条件と指標をZIPにまとめて持ち出せるようにしている。

次のセルでは、作業ディレクトリに小さなファイルを書き、読み戻せることだけを確認する。

In [ ]:
from pathlib import Path

workspace = Path.cwd() / "security-ml-lab"
workspace.mkdir(exist_ok=True)
probe = workspace / "runtime-check.txt"
probe.write_text("ready\n", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "ready\n"
probe.unlink()
print(f"Workspace: {workspace}")

## §1.3 機械学習によるモデル開発の進め方

本書の図1-8（p.11）は、データセットの作成からデプロイまでを並べ、
特徴量の検討とモデルの訓練・評価を繰り返す流れを示している。下図は、その関係をこの実習用に描き直したものだ。

![データの作成、前処理、特徴量の検討、学習と評価、運用の関係](https://raw.githubusercontent.com/black-leg-nameko/security-ml-sakuraiLab/main/notebooks/figures/01_workflow.png)

*図1　図1-8を参考にした開発工程。矢印を戻る場合も、最終テストを調整の手掛かりに使い続けない。*

各工程で、何を決めているのかをメール分類の例に置き換えると分かりやすい。

| 工程 | メール分類で決めること |
|---|---|
| データの作成 | どのメールを集め、何を根拠にspam/hamを付けるか |
| 読み込み・前処理 | 空の本文、重複、文字コードの違いをどう扱うか |
| 探索・特徴量設計 | 単語の出現、URL、ヘッダーなど、どの情報を入力にするか |
| 訓練と評価 | 分類器の種類、探索範囲、分割方法、評価指標をどう定めるか |
| 運用 | 隔離・警告・人手確認のどれに使い、誤判定をどう調べるか |

### 補足：データを式で表す

サンプルが$n$件、特徴量が$d$個あるとき、1件の入力を$x_i$、その正解を$y_i$と書く。
表形式のデータなら、入力をまとめた行列は$X\in\mathbb{R}^{n\times d}$となる。
第2章では、危険側を$y_i=1$、正常側を$y_i=0$に揃える。

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n},\qquad
s_i=f_\theta(x_i),\qquad
\widehat y_i=\mathbf{1}[s_i\geq\tau]
$$

$f_\theta$は学習するモデル、$\theta$は学習によって決まる値、$s_i$は出力スコア、$\tau$は判定閾値である。
モデルの学習と、どのスコアから警告するかを決めることは、別の操作になる。
この記法は実装を読むための補足であり、本書第1章で数式の導出を行っているわけではない。

### 小さな動作確認

次のセルでは合成データを使い、訓練とテストを分けてロジスティック回帰を学習する。
実メールや攻撃データを使っていないので、ここでの正解率はセキュリティ上の検知性能を意味しない。

`Pipeline`の中に標準化と分類器を並べると、`fit(X_train, y_train)`で標準化の平均・分散も訓練部分から求められる。
テスト時には、その値を使って変換する。テストデータ自身の平均・分散で学習し直さない。

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=10, n_informative=6, random_state=101)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=101,
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=101))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f"Synthetic holdout accuracy: {accuracy_score(y_test, prediction):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, prediction, colorbar=False)
plt.show()

### 出力をどう読むか

表示される正解率は、学習に使わなかった100件に対して、予測が正解と一致した割合である。
混同行列では、行が実際のクラス、列が予測クラスになっている。
対角成分は正解、対角以外は誤判定であり、第2章ではその内訳をFPとFNに分けて読む。

実習で使うメソッドは、次の3つを区別すると追いやすい。

| 呼び出し | ここで行っていること |
|---|---|
| `model.fit(X_train, y_train)` | 訓練データから前処理の値と分類器のパラメータを学習する |
| `model.predict(X_test)` | 学習済みの値を固定して、未知データのクラスを求める |
| `accuracy_score(y_test, prediction)` | 予測と、評価者が持つ正解ラベルを照合する |

本書第1章には練習問題がない（§1.5、p.12）。ゼミでは、次の2点を確認して第2章へ進むとよい。

1. ランタイムを作り直したときに失われるものを、ノートブック、変数、ファイルに分けて説明できるか。
2. テストデータでも`fit`してから評価すると、何を調べたことになるか。

2の答えは、未知データへの性能を測るはずの入力を学習に使ってしまう、ということだ。
第2章のTF-IDFでも同じ問題が起きる。単語の辞書やIDFの計算も、学習の一部として扱う。

**参照**：本書第1章、pp.1–12。数式、表、図の配置と確認事項はゼミ用に補った。
第2章のノートブックは、このノートブックを先に実行しなくても単独で動く。